# The campaign threshold Λ_c: closed form against simulation (Sects. 5.1, 5.2)

**In the paper:** Sect. 5.1 (the threshold; its analytic approximation, Vitells & Gross 2011; "≲ 1 %"); Sect. 5.2 ("We adopt Λ_c = 19.1 … fewer than 1 % of the searches exceed it").

A candidate on the sky is accepted when the joint fit with it improves χ² by
more than Λ_c over the fit without it (Sect. 5.1). Λ_c is the level that
noise alone exceeds in 1 % of campaigns, for a candidate searched anywhere in
the field: the map's box (±600 mas) minus the primary's 30 mas zone, and every
relative proper motion within ±10 mas/yr per component.

Under the null the improvement at a trial point (x, μ) is max(Z, 0)², with Z
the candidate's best-fit flux over its error: a standard Gaussian field over
the four-dimensional searched region. The chance that its maximum exceeds t
is approximated in closed form by the expected Euler characteristic of the
excursion set, an approximation that improves as t grows,

$$P(\max Z > t) \approx \sum_d L_d\,\rho_d(t),\qquad
\rho_d(t) = (2\pi)^{-(d+1)/2}\,\mathrm{He}_{d-1}(t)\,e^{-t^2/2},$$

with L_d the curvatures of the region in the field's own metric, which follow
from the instrument and the scan law alone (`thr.search_curvatures`,
`thr.lrt_threshold`). This notebook computes Λ_c in closed form and checks it
against simulated noise-only campaigns (`sim.field_null_max`).

**The configurations.** Figure 4's law (the 47 GOST transits at the Pleiades
centre), with the primary pinned or free in the fits, and with two noise
models: the same noise on every sample (uniform), or Figure 4's per-sample
model (G = 16, the quiet background, the read noise). The closed form assumes
uniform noise.

| step | task | calls | paper |
|---|---|---|---|
| 1 | the scan law and Fig. 4's noise | `scanlaw.load_scan_law`, `noise.gaia_like`, `sim.random_zero_points`, `ins.render_window` | Sect. 5.2 |
| 2 | Λ_c in closed form, from the curvatures of the searched region | `thr.search_curvatures`, `thr.lrt_threshold` | Sect. 5.1 |
| 3 | the simulated null: the strongest candidate of 3200 noise-only campaigns per configuration | `sim.field_null_max` | Sect. 5.1 |
| 4 | the rate at Λ_c, and the simulated 1 % level | — | Sects. 5.1, 5.2 |
| 5 | the survival curves | — | — |

Every scanlib call carries the name of its module (`dyn.wls`, `joint.fit_joint`); the first code cell lists the
modules.

**What it reproduces.** Λ_c = 19.13 for Figure 4 (law and zero points, primary
free), the paper's 19.1. Simulated, 8 × 400 campaigns per configuration: with
uniform noise and a free primary 32 of 3200 searches exceed the bar (1.00 %,
exact 95 % interval 0.68–1.41 %); with Figure 4's noise and a free primary 24
of 3200 (0.75 %, 0.48–1.11 %). The rate at the closed-form bar is consistent with
1 % or less in all four configurations: the lower end of every interval is below
1 %, and the upper ends, 1.11 to 1.41 %, do not exclude a rate a little above it.
The closed form is the most permissive case, a
candidate anywhere and no further gate.

**Run time.** The 12 800 simulated campaigns take about 12 hours (3–5 s
each), so they are stored in `../stored/lambda_c_null.npz`. By default the
notebook loads them and recomputes the first campaign of each configuration,
which must equal the stored one (bit for bit; to 1e-12 for the pinned,
uniform configuration, whose records come from an earlier script that writes
the same projection in scalar form). About 30 s. `FULL = True` recomputes all
of them and rewrites the store.

In [ ]:
# the imports: numpy, matplotlib, scipy's beta distribution, and the scanlib modules; every scanlib call below
# carries its module's name
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta

from scanlib import instrument as ins      # the instrument: sampling, LSF, the along-scan projection (Eq. 9)
from scanlib import noise                  # the noise model of a window
from scanlib import scanlaw                # scan angles, epochs and parallax factors
from scanlib import simulation as sim      # synthetic scenes and their windows
from scanlib import thresholds as thr      # the threshold of the search, Lambda_c

In [ ]:
# the instrument, defined once (Sect. 2.1): 18 samples of 59 mas along scan, a Gaussian LSF of FWHM 100 mas,
# and its per-window margins Delta_1, Delta_2 (App. A.2; derived in sim_margins_lambda_w)
FWHM_MAS, PITCH_MAS, N_SAMPLES = 100.0, 59.0, 18
inst = ins.Instrument.gaussian(fwhm_mas=FWHM_MAS, pitch_mas=PITCH_MAS, n_samples=N_SAMPLES, margins=ins.TOY_MARGINS)

# the switch FULL and the store of the simulated null
FULL = False                               # True: recompute the 12 800 campaigns (about 12 h)
STORE = Path("..") / "stored" / "lambda_c_null.npz"

## 1. The law and Figure 4's noise

The noise of a sample follows the paper's Eq. 21 for a primary of G = 16
whose transit sums nine CCD crossings (Sect. 5.2): in electrons,
σ² = signal + 9 (b + r²), with b the background and r the read noise of one
crossing. `noise.gaia_like` computes it in electrons and returns it in the
toy's flux units (σ divided by the electrons per flux unit). The windows of Figure 4 sit at random sampling phases,
their zero points, drawn from Figure 4's stream. `scanlaw.load_scan_law` returns the epochs already in years from
the mean epoch, the reference epoch, so `taus` below equals `t` to rounding.

In [ ]:
# Figure 4's scan law, and its epochs in years from the mean epoch
# psi: the scan angle of each transit [rad]; t: its epoch [yr from the reference epoch]
psi, t = scanlaw.load_scan_law(scanlaw.PLEIADES)
taus = t - t.mean()                        # years from the mean epoch; t is that already, so taus = t to rounding

In [ ]:
# Figure 4's per-sample noise model (a G = 16 primary, nine CCD crossings), and its zero points
PRIMARY_G = 16.0                           # the primary's magnitude
BACKGROUND_E_PER_SAMPLE_PER_CCD = 10.0     # the background [electrons per sample per CCD crossing]
# noise.gaia_like: Eq. 21, in electrons sigma^2 = signal + 9 (b + r^2); returned per sample in the toy's flux units
NOISE_MODEL = noise.gaia_like(PRIMARY_G, sim.PRIMARY_FLUX, background=BACKGROUND_E_PER_SAMPLE_PER_CCD)
# the generator of the zero points. Its seed is the one of the Figure 4 notebook: [Figure 4's seed, 3], where 3
# is the number of the zero points' own stream there
rng_zero_points = np.random.default_rng([20260929, 3])
# sim.random_zero_points: one sampling phase (zero point eta0, mas) per window
ZERO_POINTS = sim.random_zero_points(rng_zero_points, psi.size, inst)   # Figure 4's

In [ ]:
# the noise of a window that holds the primary alone: what the null whitens by
# (ins.render_window: the noise-free samples of a window; noise.NoiseModel.sigma: their per-sample noise)
# the window: one source, the primary, at the window's centre (0 mas) with the primary's flux
primary_window = ins.render_window([0.0], [sim.PRIMARY_FLUX], inst)
SIGMA_FIG4 = NOISE_MODEL.sigma(primary_window)             # one sigma per sample, in the toy's flux units
print(f"{psi.size} transits; |tau| <= {np.abs(taus).max():.2f} yr")
print(f"Figure 4 noise per sample: {SIGMA_FIG4.min():.3f} in the wings to {SIGMA_FIG4.max():.3f} "
      f"at the peak (primary flux {sim.PRIMARY_FLUX:.0f})")

## 2. The threshold in closed form (Sect. 5.1)

`thr.search_curvatures` gives the curvatures L_0 … L_4 of the searched region;
`thr.lrt_threshold` solves Σ L_d ρ_d(t) = 1 % for u = t². With a free primary the
candidate's template is made orthogonal to the primary's flux, offset and
residual motion, and the level barely moves. The bar of Figure 4 is the free
primary's at its zero points.

In [ ]:
# the closed form, first part: the curvatures of the searched region, with the primary free and pinned
# thr.search_curvatures: L_0..L_4 of the searched region (positions and motions) in the field's own metric
L_FREE = thr.search_curvatures(psi, inst, taus, free_primary=True)
L_PIN = thr.search_curvatures(psi, inst, taus)
print("curvatures L_0..L_4, free primary:", np.round(L_FREE, 2))

In [ ]:
# Lambda_c with the primary pinned and free, with zero points 0 and with Figure 4's; then Figure 4's bar
# thr.lrt_threshold: Lambda_c, the u = t^2 at which sum_d L_d rho_d(t) equals the false rate (1 %)
# the two fits of the primary: its printed name, and whether the primary is free
for tag, free_primary in (("pinned", False), ("free", True)):
    level_zero = thr.lrt_threshold(psi, inst, taus, free_primary=free_primary)
    level_fig4 = thr.lrt_threshold(psi, inst, taus, free_primary=free_primary, eta0=ZERO_POINTS)
    print(f"Lambda_c, primary {tag:6s}: {level_zero:.3f} "
          f"(zero points 0), {level_fig4:.3f} "
          f"(Figure 4's zero points)")
BAR = thr.lrt_threshold(psi, inst, t, free_primary=True, eta0=ZERO_POINTS)   # t = taus to rounding (section 1)
print(f"Figure 4's bar: Lambda_c = {BAR:.2f}")
assert round(BAR, 2) == 19.13              # the paper's 19.1

In [ ]:
# the closed form's survival curve, which section 5 draws (closed_form)
def closed_form(L, u):
    """The closed form of the chance that the strongest candidate of a noise-only search exceeds a level:
    P(max Z > t) = sum over d of L_d rho_d(t), with rho_d(t) = (2 pi)^(-(d+1)/2) He_{d-1}(t) exp(-t^2 / 2).

    Inputs
        L   the curvatures (L_0, ..., L_D) of the searched region, from thr.search_curvatures; L_0 is not used (the
            sum starts at d = 1; L_0 = 0 for these regions)
        u   the level of Lambda = max(Z, 0)^2, a number or an array; the level of Z is t = sqrt(u)
    Returns the chance (the same shape as u). It is an approximation for high levels."""
    t = np.sqrt(u)                         # the level of Z
    # he holds the Hermite polynomials He_0..He_3 of t; he[d - 1] is the He_{d-1} of the term of dimension d (so D is 4 at most)
    he = (np.ones_like(t), t, t ** 2 - 1, t ** 3 - 3 * t)
    # the sum over the dimensions d = 1, ..., D of the searched region; len(L) is D + 1
    chance = 0
    for d in range(1, len(L)):
        chance = chance + L[d] * (2 * np.pi) ** (-(d + 1) / 2) * he[d - 1] * np.exp(-t ** 2 / 2)
    return chance

## 3. The simulated null

`sim.field_null_max` draws whitened noise in every window, removes its projection
on the primary's columns, correlates the rest with the whitened candidate
template over the searched region (a grid of 6 mas and 3 mas/yr, refined by
Nelder–Mead), and returns the maximum of Z per campaign. Seed s draws the
same noise in every configuration, so the configurations compare paired.
The zero points are 0 in the simulation; the bar barely depends on them (§2).

In [ ]:
# the simulated null: max Z per campaign for 4 configurations x 8 seeds x 400 campaigns
# a configuration's name is "<the noise>_<the primary>": the noise uniform or Figure 4's, the primary pinned or free
CONFIGS = ("uniform_pinned", "uniform_free", "fig04_pinned", "fig04_free")
SEEDS = np.arange(101, 109)                # the eight seeds: 101, 102, ..., 108
N_PER_SEED = 400                           # the number of campaigns of each seed

In [ ]:
# the first n campaigns of one configuration and seed (run)
def run(config, seed, n):
    """The strongest candidate of each of the first n noise-only campaigns of one configuration and seed.

    Inputs
        config   the configuration's name, "<the noise>_<the primary>": "uniform" or "fig04", "pinned" or "free"
        seed     the seed of the noise, a whole number
        n        the number of campaigns
    Returns n numbers: max Z of each campaign, from sim.field_null_max (the strongest candidate's best-fit flux
    over its error, searched over the whole region, in noise-only windows)."""
    # step 1, the configuration: the text before the "_" is the noise, the text after it is the primary
    noise_kind, primary = config.split("_")
    free_primary = primary == "free"
    if noise_kind == "fig04":
        sigma_noise = SIGMA_FIG4           # Figure 4's per-sample noise
    else:
        sigma_noise = None                 # the same noise on every sample
    # step 2, the generator of the noise: its seed is the seed alone (the configuration is not in it), so a seed
    # draws the same noise in every configuration
    rng_noise = np.random.default_rng(seed)
    # step 3, the campaigns
    return sim.field_null_max(psi, taus, n, rng_noise, inst, free_primary=free_primary, sigma_noise=sigma_noise)

In [ ]:
# the simulated null: recomputed and stored with FULL, otherwise loaded from the store
# max_z: max Z of every campaign, an array over (configuration, seed, campaign)
if FULL:
    max_z_rows = []                        # one row per configuration
    for config in CONFIGS:
        max_z_of_config = []               # one entry per seed: the N_PER_SEED campaigns of that seed
        for seed in SEEDS:
            max_z_of_config.append(run(config, seed, N_PER_SEED))
        max_z_rows.append(max_z_of_config)
    max_z = np.array(max_z_rows)
    STORE.parent.mkdir(exist_ok=True)
    np.savez(STORE, max_z=max_z, configs=np.array(CONFIGS), seeds=SEEDS)
    print(f"computed and stored {STORE}")
else:
    with np.load(STORE) as stored:
        max_z = stored["max_z"]
        # the store was made with this notebook's configurations and seeds
        assert tuple(stored["configs"]) == CONFIGS and np.array_equal(stored["seeds"], SEEDS)
    print(f"loaded {STORE}")
assert max_z.shape == (len(CONFIGS), SEEDS.size, N_PER_SEED)

In [ ]:
# the first campaign of each configuration recomputed and compared with the store. max_z is (configuration, seed,
# campaign): [i, 0, :1] keeps configuration i, the first seed and the first campaign, as an array of one element
for i, config in enumerate(CONFIGS):       # i: the configuration's number
    fresh = run(config, SEEDS[0], 1)
    stored_first = max_z[i, 0, :1]
    if config == "uniform_pinned":
        # equal to 1e-12: this configuration's records come from an earlier script (see the introduction)
        assert np.allclose(fresh, stored_first, rtol=0, atol=1e-12), config
    else:
        assert np.array_equal(fresh, stored_first), config
    print(f"{config:15s} campaign 0 of seed {SEEDS[0]}: max Z = {fresh[0]:.6f}, as stored")

## 4. The rate at the bar, and the simulated 1 % level (Sects. 5.1, 5.2)

The rate is the fraction of searches whose maximum exceeds the bar, with its
exact (Clopper–Pearson) 95 % interval; the simulated 1 % level is the 99th
percentile of max(Z)², with a bootstrap 95 % interval (2000 resamples).

In [ ]:
# the rate k/n of k searches above the bar among n, with its exact interval (rate)
def rate(k, n):
    """The rate of k searches above a level among n, with its exact (Clopper-Pearson) 95 % interval, from the
    beta distribution.

    Inputs
        k   the number of searches above the level
        n   the number of searches
    Returns three numbers
        the rate k / n
        the lower end of its 95 % interval; 0 when k = 0
        the upper end of its 95 % interval; 1 when k = n"""
    if k > 0:
        lower = beta.ppf(0.025, k, n - k + 1)
    else:
        lower = 0.0
    if k < n:
        upper = beta.ppf(0.975, k + 1, n - k)
    else:
        upper = 1.0
    return k / n, lower, upper

In [ ]:
# per configuration: the rate above the bar with its exact interval, and the simulated 1 % level with a
# bootstrap interval
N_BOOTSTRAP = 2000                         # the number of resamples of the bootstrap
# the generator of the bootstrap: one stream for the four configurations, in the order of CONFIGS. The seed is 0;
# the reason for that value: not stated
rng = np.random.default_rng(0)
# R: a table from a configuration's name to its results, itself a table with six entries
#     "lam"       Lambda = (max Z)^2 of each of its 3200 campaigns
#     "k"         the number of campaigns with Lambda above the bar
#     "n"         the number of campaigns
#     "rate"      (k / n, the lower end, the upper end of its exact 95 % interval)
#     "level"     the simulated 1 % level: the 99th percentile of Lambda
#     "level_ci"  (the lower end, the upper end) of the level's bootstrap 95 % interval
R = {}
for i, config in enumerate(CONFIGS):       # i: the configuration's number
    # step 1, Lambda of every campaign
    # max_z[i].ravel(): configuration i, its 8 seeds x 400 campaigns as one vector of 3200;
    # np.clip(..., 0, None) is max(Z, 0): a negative max Z counts as Lambda = 0 (none occurs in the store)
    lam = np.square(np.clip(max_z[i].ravel(), 0.0, None))
    n = lam.size
    # step 2, the bootstrap of the 1 % level: n campaigns drawn with replacement from the n, and their 99th
    # percentile, N_BOOTSTRAP times
    boot = []
    for _ in range(N_BOOTSTRAP):           # one pass per resample; the counter is not used
        resample = rng.choice(lam, n)
        boot.append(np.quantile(resample, 0.99))
    # step 3, the rate above the bar and the level, with their intervals
    k = int((lam > BAR).sum())
    level_interval = (np.percentile(boot, 2.5), np.percentile(boot, 97.5))
    R[config] = dict(lam=lam, k=k, n=n, rate=rate(k, n), level=np.quantile(lam, 0.99),
                     level_ci=level_interval)
    fraction, lower, upper = R[config]["rate"]
    print(f"{config:15s} {k:3d} of {n} above {BAR:.2f}: {100 * fraction:.2f} % "
          f"[{100 * lower:.2f}, {100 * upper:.2f}]; "
          f"simulated 1 % level {R[config]['level']:.2f} [{level_interval[0]:.2f}, {level_interval[1]:.2f}]")

In [ ]:
# Sect. 5.1, 5.2: consistent with 1 % or less in every configuration (the lower end
# of each 95 % interval below 1 %); 1.00 % for uniform noise
# with a free primary, 0.75 % for Figure 4's noise with a free primary
# rate returns (the rate, the interval's lower end, its upper end): [1] keeps the lower end
for config in CONFIGS:
    assert R[config]["rate"][1] < 0.01, config
# the counts above the bar, of 3200: 32 is 1.00 %, 24 is 0.75 %
assert (R["uniform_free"]["k"], R["fig04_free"]["k"]) == (32, 24)
assert (R["uniform_pinned"]["k"], R["fig04_pinned"]["k"]) == (27, 25)
print("every rate at the closed-form bar is consistent with 1 % or less")

## 5. The survival curves

The fraction of null searches whose strongest candidate exceeds u, per
configuration, against the closed form for uniform noise (pinned and free
nearly coincide). The vertical line is Figure 4's bar, the horizontal one
1 %.

In [ ]:
# the figure: the simulated survival curves of the four configurations against the closed form
PAPER_STYLE = {"font.size": 10.0, "xtick.labelsize": 9.0, "ytick.labelsize": 9.0, "axes.linewidth": 0.5,
               "xtick.direction": "in", "ytick.direction": "in"}
with plt.rc_context(PAPER_STYLE):                                         # the paper's style, this plot only
    u = np.linspace(10, 26, 300)           # the levels of Lambda at which the closed form is drawn
    fig, ax = plt.subplots(figsize=(6.0, 4.2))
    # styles: a table from a configuration's name to its line's (colour, dash): gray for uniform noise, black
    # for Figure 4's; dashed for the pinned primary, solid for the free one
    styles = {"uniform_pinned": ("0.6", "--"), "uniform_free": ("0.6", "-"),
              "fig04_pinned": ("k", "--"), "fig04_free": ("k", "-")}
    for config in CONFIGS:
        lam = np.sort(R[config]["lam"])    # Lambda of the campaigns, in increasing order
        # the fraction of the campaigns above each sorted Lambda: 1 - (its rank) / (the number of campaigns)
        surv = 1.0 - np.arange(1, lam.size + 1) / lam.size
        colour, dash = styles[config]
        ax.step(lam, surv, where="post", color=colour, ls=dash, lw=1.0,
                label=config.replace("_", ", ") + " (simulated)")
    ax.plot(u, closed_form(L_FREE, u), color="k", lw=1.4, ls="-.", label="closed form, uniform, free")
    ax.axvline(BAR, color="0.3", lw=0.6, ls=":")           # Figure 4's bar
    ax.axhline(0.01, color="0.3", lw=0.6, ls=":")          # 1 %
    ax.set_yscale("log")
    ax.set_xlim(u[0], u[-1])               # u[0], u[-1]: the first and the last level
    ax.set_ylim(2e-4, 1.0)
    ax.set_xlabel(r"$u$ (improvement $\Delta\chi^2$)")
    ax.set_ylabel(r"fraction of null searches with $\Lambda_{\max} > u$")
    ax.tick_params(direction="in", which="both")
    ax.legend(fontsize=7, loc="lower left")
    fig.tight_layout()
    plt.show()